# 03 - 마스크율과 Monte Carlo 분산

**학습 목표**: 기대값을 보정하는 `1/t` estimator가 작은 `t`에서 왜 noisy한지 simulation합니다. 이는 논문의 실제 loss curve 재현이 아닙니다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 라이브러리만 사용합니다.

In [ ]:
import random
import statistics

def estimate(sequence_length, t, rng):
    masked = sum(rng.random() < t for _ in range(sequence_length))
    # 각 token의 가상 loss를 1로 두면 기대값은 sequence_length입니다.
    return masked / t

rng = random.Random(11)
for t in (0.02, 0.10, 0.50, 0.90):
    samples = [estimate(128, t, rng) for _ in range(5000)]
    print(f't={t:0.2f} mean={statistics.mean(samples):6.2f} variance={statistics.pvariance(samples):8.2f}')
    assert abs(statistics.mean(samples) - 128) < 5

In [ ]:
def forward_budget(length, diffusion_steps):
    return {
        'AR generation': length,
        'full-sequence diffusion': diffusion_steps,
    }

for steps in (8, 16, 32, 64):
    budget = forward_budget(128, steps)
    print(steps, budget)

print('주의: forward 수가 적어도 양방향 attention 재계산 비용 때문에 wall-clock이 자동으로 더 빠르지는 않습니다.')

## 결론

낮은 mask ratio는 estimator 평균을 바꾸지 않지만 분산을 크게 만듭니다. 또한 NFE와 실제 latency는 다릅니다. 이 두 구분은 후속 Block Diffusion과 self-speculation 논문을 읽을 때 핵심입니다.